# Statistical Analysis of the Toxicity Studies (Subacute, Subchronic and Reversibility)

One-way ANOVA with Tukey HSD post-hoc comparisons for every haematological and biochemical parameter
in each study, followed by a cross-study summary.

## Inputs
The raw data sheets are in `data/raw/` (parameters in rows, one block of readings per group; Mean/SD columns are optional). Converted long-format files are written to `data/processed/` and all results to `results/`:

| Study | Raw sheet (wide) | Converted file (long, created automatically) |
|---|---|---|
| Subacute (28-day) | `Subacute_Toxicity_Raw_Data.csv` | `Toxicity_subacute_anova_file.csv` |
| Subchronic (60-day) | `Subchronic_Toxicity_Raw_Data.csv` | `Toxicity_subchronic_anova_file.csv` |
| Reversibility | `Reversibility_Toxicity_Raw_Data.csv` | `Toxicity_reversibility_anova_file.csv` |

## Methods
- Wide-to-long conversion with a check that the recomputed mean and SD match the sheet's Mean/SD columns
- One-way ANOVA per parameter (`value ~ group`), eta squared
- Tukey HSD for all pairwise group comparisons (`meandiff` = second group minus first group in each comparison)
- Assumption checks: Shapiro-Wilk (residuals) and Levene; Kruskal-Wallis is reported alongside as a nonparametric check
- **Significance convention: `*` = p < 0.05** (single level; no ** or ***)

Corresponding author: Abdullahi Abubakar Sadiq (abubakarabdulsadiq@gmail.com)

In [ ]:
# =========================================================
# SETUP
# =========================================================
import os, re, csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import shapiro, levene, kruskal

from pathlib import Path
# Repository paths (works when run from the repository root or from notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

ALPHA = 0.05   # * = p < 0.05

STUDIES = {
    "Subacute (28-day)":   {"raw": str(DATA / "raw" / "Subacute_Toxicity_Raw_Data.csv"),
                            "long": str(DATA / "processed" / "Toxicity_subacute_anova_file.csv"),   "prefix": str(RESULTS / "Subacute")},
    "Subchronic (60-day)": {"raw": str(DATA / "raw" / "Subchronic_Toxicity_Raw_Data.csv"),
                            "long": str(DATA / "processed" / "Toxicity_subchronic_anova_file.csv"), "prefix": str(RESULTS / "Subchronic")},
    "Reversibility":       {"raw": str(DATA / "raw" / "Reversibility_Toxicity_Raw_Data.csv"),
                            "long": str(DATA / "processed" / "Toxicity_reversibility_anova_file.csv"), "prefix": str(RESULTS / "Reversibility")},
}

def format_p(p):
    if pd.isna(p):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"

def star(p):
    return "*" if (not pd.isna(p) and p < ALPHA) else ""

def is_control(g):
    return "control" in str(g).lower()

def group_sort_key(g):
    """Treated groups by ascending dose, control last."""
    m = re.search(r"(\d+(?:\.\d+)?)\s*mg", str(g))
    return (is_control(g), float(m.group(1)) if m else 0.0, str(g))

def clean_group(name):
    name = re.sub(r"\s+", " ", name.replace("\n", " ")).strip()
    if is_control(name):
        return "Control"
    name = re.sub(r"\s*Treatment Group", "", name, flags=re.I)
    return re.sub(r"(\d)\s*mg", r"\1 mg", name)       # "40mg/kg" -> "40 mg/kg"

def clean_text(s):
    s = re.sub(r"[^\x20-\x7e]", " ", s)                 # drop odd legacy-encoding characters
    return re.sub(r"\s+", " ", s).strip()

In [ ]:
# =========================================================
# WIDE -> LONG CONVERSION  (parameter, reference_range, group, replicate, value)
# =========================================================
# Canonical parameter names (the studies label a few tests differently)
CANONICAL = {
    "HDLC": "High Density Lipoprotein (HDL)",
    "LDL-C": "Low Density Lipoprotein (LDL)",
    "ALT": "Alanine transaminase (ALT) test",
    "AST": "Aspartate transaminase (AST)",
}

def canonical_name(name):
    name = clean_text(name)
    abbr = re.findall(r"\(([^)]+)\)", name)
    return CANONICAL.get(abbr[-1].strip(), name) if abbr else name

def wide_to_long(path):
    """Reads a wide sheet (with or without Mean/SD columns) and returns long format."""
    with open(path, encoding="cp1252", newline="") as f:
        rows = list(csv.reader(f))

    # READING row = replicate labels; the row above holds the group names
    ridx = next(i for i, r in enumerate(rows)
                if r and r[0].strip().upper().startswith("READING"))
    header, reading = rows[ridx - 1], rows[ridx]

    starts = [j for j, c in enumerate(header)
              if j >= 2 and c.strip() and not re.search(r"anova|tukey|turkey", c, re.I)]
    ends = starts[1:] + [len(reading)]

    blocks = []                         # (group, reading columns, mean col or None, sd col or None)
    for s, e in zip(starts, ends):
        cols, mean_col, sd_col = [], None, None
        for j in range(s, e):
            c = reading[j].strip().lower() if j < len(reading) else ""
            if c == "mean":
                mean_col = j
            elif c == "sd":
                sd_col = j
            elif c == "":
                break
            else:
                cols.append(j)
        blocks.append((clean_group(header[s]), cols, mean_col, sd_col))

    out, mismatches = [], []
    for r in rows[ridx + 1:]:
        if not r or not r[0].strip():
            continue
        try:
            vals_by_group = [[float(r[j]) for j in cols] for _, cols, _, _ in blocks]
        except (ValueError, IndexError):
            continue                    # label or footer rows
        param, ref = canonical_name(r[0]), clean_text(r[1])
        for (g, cols, mc, sc), vals in zip(blocks, vals_by_group):
            if mc is not None and sc is not None:     # check against the sheet's Mean/SD, if present
                mu, sd = float(r[mc]), float(r[sc])
                if abs(np.mean(vals) - mu) > 0.011 or abs(np.std(vals, ddof=1) - sd) > 0.011:
                    mismatches.append((param, g, round(np.mean(vals), 2), mu,
                                       round(np.std(vals, ddof=1), 2), sd))
            for i, v in enumerate(vals, 1):
                out.append([param, ref, g, i, v])

    long_df = pd.DataFrame(out, columns=["parameter", "reference_range", "group", "replicate", "value"])
    return long_df, mismatches

# Convert every raw sheet that is present
for label, cfg in STUDIES.items():
    if os.path.exists(cfg["raw"]):
        long_df, mism = wide_to_long(cfg["raw"])
        long_df.to_csv(cfg["long"], index=False)
        n_par = long_df["parameter"].nunique()
        n_per = long_df.groupby(["parameter", "group"]).size().iloc[0]
        print(f"{label}: {n_par} parameters, {long_df['group'].nunique()} groups, n = {n_per} per group -> {cfg['long']}")
        if mism:
            print("  WARNING: recomputed mean/SD differs from the sheet for:")
            for m in mism:
                print("   ", m)
        else:
            print("  No mismatches (Mean/SD columns, where present, agree with the readings).")
    else:
        print(f"{label}: raw sheet '{cfg['raw']}' not found (will use the long file if present).")

In [ ]:
# =========================================================
# ONE-WAY ANOVA + TUKEY HSD FOR ONE STUDY
# =========================================================
def run_study(label, d, prefix):
    d = d.copy()
    order = sorted(d["group"].unique(), key=group_sort_key)
    d["group"] = pd.Categorical(d["group"], categories=order, ordered=True)

    treated = [g for g in order if not is_control(g)]
    short = {g: f"T{i + 1}" for i, g in enumerate(treated)}
    short.update({g: "Control" for g in order if is_control(g)})

    anova_rows, tukey_rows = [], []
    for param, sub in d.groupby("parameter", sort=False):
        sub = sub.dropna(subset=["value"])
        samples = [g["value"].values for _, g in sub.groupby("group", observed=True)]

        model = ols("value ~ C(group)", data=sub).fit()
        aov = sm.stats.anova_lm(model, typ=2)
        ss_resid = aov.loc["Residual", "sum_sq"]

        if sub["value"].nunique() == 1 or ss_resid < 1e-12:
            anova_rows.append({"Parameter": param, "F": np.nan, "df": "", "p_value": np.nan,
                               "p_formatted": "n/a", "Sig": "", "eta_sq": np.nan,
                               "Shapiro_p": np.nan, "Levene_p": np.nan, "KW_p": np.nan,
                               "Assumption_flag": "",
                               "Note": "No variance; ANOVA undefined"})
            continue

        p = aov.loc["C(group)", "PR(>F)"]
        sh_p = shapiro(model.resid).pvalue if len(model.resid) >= 3 else np.nan
        lv_p = levene(*samples).pvalue
        kw_p = kruskal(*samples).pvalue
        anova_rows.append({
            "Parameter": param,
            "F": aov.loc["C(group)", "F"],
            "df": f'{int(aov.loc["C(group)", "df"])}, {int(aov.loc["Residual", "df"])}',
            "p_value": p, "p_formatted": format_p(p), "Sig": star(p),
            "eta_sq": aov.loc["C(group)", "sum_sq"] / aov["sum_sq"].sum(),
            "Shapiro_p": sh_p, "Levene_p": lv_p, "KW_p": kw_p,
            "Assumption_flag": "check" if (sh_p < ALPHA or lv_p < ALPHA) else "",
            "Note": ""})

        # Prefix labels with their rank so statsmodels' alphabetical ordering follows T1, T2, T3, Control
        coded = sub["group"].astype(str).map({g: f"{i:02d}|{g}" for i, g in enumerate(order)})
        tk = pairwise_tukeyhsd(endog=sub["value"], groups=coded, alpha=ALPHA)
        tdf = pd.DataFrame(tk._results_table.data[1:], columns=tk._results_table.data[0])
        for _, r in tdf.iterrows():
            g1, g2 = r["group1"].split("|", 1)[1], r["group2"].split("|", 1)[1]
            tukey_rows.append({
                "Parameter": param,
                "Comparison": f'{short[g1]} vs {short[g2]}',
                "meandiff": r["meandiff"], "p_adj": r["p-adj"],
                "p_formatted": format_p(r["p-adj"]), "Sig": star(r["p-adj"]),
                "lower": r["lower"], "upper": r["upper"]})

    anova_df, tukey_df = pd.DataFrame(anova_rows), pd.DataFrame(tukey_rows)

    # Manuscript table: mean +/- SD per group, * on the parameter if the ANOVA is significant
    s = (d.groupby(["parameter", "group"], sort=False, observed=True)["value"]
           .agg(["mean", "std"]).reset_index())
    s["mean_sd"] = s.apply(lambda r: f'{r["mean"]:.2f} ± {r["std"]:.2f}', axis=1)
    table = s.pivot(index="parameter", columns="group", values="mean_sd")[order]
    table = table.reindex(d["parameter"].unique())
    stars = anova_df.set_index("Parameter")["Sig"]
    table["ANOVA p"] = anova_df.set_index("Parameter")["p_formatted"]
    table.index = [f"{p}{stars.get(p, '')}" for p in table.index]

    anova_df.to_csv(f"{prefix}_anova_results.csv", index=False)
    tukey_df.to_csv(f"{prefix}_tukey_results.csv", index=False)
    table.to_csv(f"{prefix}_summary_table.csv")
    return anova_df, tukey_df, table, short

In [ ]:
# =========================================================
# RUN ALL AVAILABLE STUDIES
# =========================================================
results = {}
for label, cfg in STUDIES.items():
    if not os.path.exists(cfg["long"]):
        print(f"Skipping {label}: no data file.")
        continue
    data = pd.read_csv(cfg["long"])
    anova_df, tukey_df, table, short = run_study(label, data, cfg["prefix"])
    results[label] = {"anova": anova_df, "tukey": tukey_df, "table": table, "short": short, "data": data}

    print("=" * 80)
    print(f"{label}  |  {len(anova_df)} parameters | groups: {', '.join(short[g] + '=' + g for g in short)}")
    print("=" * 80)
    print(anova_df[["Parameter", "F", "p_formatted", "Sig", "eta_sq",
                    "Shapiro_p", "Levene_p", "KW_p", "Assumption_flag"]].round(3).to_string(index=False))

    sig = anova_df.loc[anova_df["p_value"] < ALPHA, "Parameter"]
    print(f"\nSignificant parameters (*): {len(sig)}")
    if len(sig):
        print(tukey_df[tukey_df["Parameter"].isin(sig) & (tukey_df["p_adj"] < ALPHA)]
              [["Parameter", "Comparison", "meandiff", "p_formatted", "Sig"]].round(3).to_string(index=False))
    print()

In [ ]:
# =========================================================
# CROSS-STUDY SUMMARY OF SIGNIFICANT PARAMETERS (* = p < 0.05)
# =========================================================
if results:
    all_anova = pd.concat([r["anova"].assign(Study=lab) for lab, r in results.items()], ignore_index=True)
    all_anova.to_csv(RESULTS / "Toxicity_all_studies_anova.csv", index=False)

    all_tukey = pd.concat([r["tukey"].assign(Study=lab) for lab, r in results.items()], ignore_index=True)
    all_tukey.to_csv(RESULTS / "Toxicity_all_studies_tukey.csv", index=False)

    wide = (all_anova.assign(txt=lambda x: x["p_formatted"] + x["Sig"])
                     .pivot(index="Parameter", columns="Study", values="txt"))
    wide = wide.reindex(all_anova["Parameter"].unique())
    wide.to_csv(RESULTS / "Toxicity_all_studies_pvalue_overview.csv")

    sig_any = all_anova[all_anova["p_value"] < ALPHA][["Study", "Parameter", "F", "p_formatted", "Sig"]]
    print("Significant parameters by study")
    print(sig_any.round(3).to_string(index=False))
    print("\nP-value overview (rows = parameters, * = p < 0.05)")
    print(wide.to_string())
else:
    print("No study data found. Add the raw sheets to the notebook folder and re-run.")

In [ ]:
# =========================================================
# FIGURES: mean +/- SD for the significant parameters of each study
# =========================================================
for label, r in results.items():
    sig = r["anova"].loc[r["anova"]["p_value"] < ALPHA, "Parameter"].tolist()
    if not sig:
        print(f"{label}: no significant parameters to plot.")
        continue
    order = list(r["short"].keys())
    ncol = min(3, len(sig)); nrow = int(np.ceil(len(sig) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.6 * nrow), squeeze=False)
    for ax, param in zip(axes.ravel(), sig):
        sub = r["data"][r["data"]["parameter"] == param]
        st = sub.groupby("group")["value"].agg(["mean", "std"]).reindex(order)
        ax.bar([r["short"][g] for g in order], st["mean"], yerr=st["std"], capsize=4)
        p = r["anova"].set_index("Parameter").loc[param, "p_formatted"]
        ax.set_title(f"{param.split('(')[-1].rstrip(')') if '(' in param else param}*\n(ANOVA p {p})", fontsize=10)
        ax.set_ylabel("Mean ± SD")
    for ax in axes.ravel()[len(sig):]:
        ax.axis("off")
    fig.suptitle(f"{label}: parameters significant at p < 0.05 (*)", y=1.02)
    fig.tight_layout()
    fig.savefig(f"{STUDIES[label]['prefix']}_significant_parameters.png", dpi=300, bbox_inches="tight")
    plt.show()

print("Done. All tables (.csv) and figures (.png) are saved in the working folder.")